In [ ]:
import torch
from torch import nn
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from skimage.metrics import structural_similarity as ssim
import os


In [ ]:
# Dataset Preparation

transform = transforms.Compose([
    transforms.ToTensor()
])

dataset = MNIST(root="./data", train=True, transform=transform, download=True)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Diffusion Settings

T = 50
device = "cuda" if torch.cuda.is_available() else "cpu"
betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = (1 - betas).to(device)
alphas_bar = torch.cumprod(alphas, dim=0).to(device)

def forward_diffusion_sample(x_0, t):
    noise = torch.randn_like(x_0)
    t = t.to(device)  # Ensure t is on the same device
    sqrt_alpha_bar = torch.sqrt(alphas_bar[t])[:, None, None, None]
    sqrt_one_minus_alpha_bar = torch.sqrt(1 - alphas_bar[t])[:, None, None, None]
    return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise, noise



In [ ]:
# Transformer Model

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        x = x.view(B, C, -1).permute(0, 2, 1)
        attn_output, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_output)
        ff_output = self.ff(x)
        x = self.norm2(x + ff_output)
        return x.permute(0, 2, 1).view(B, C, H, W)

class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU()
        )
        self.transformer = TransformerBlock(64, num_heads=4)
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 3, padding=1)
        )

    def forward(self, x, t):
        x = self.encoder(x)
        x = self.transformer(x)
        x = self.decoder(x)
        return x



In [ ]:
# Training Loop

model = Denoiser().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
mse_loss = nn.MSELoss()

def train(model, dataloader, epochs=10):
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, _ in tqdm(dataloader):
            x = x.to(device)
            t = torch.randint(0, T, (x.size(0),), device=device).long()
            noisy, noise = forward_diffusion_sample(x, t)
            noise_pred = model(noisy, t)
            loss = mse_loss(noise_pred, noise)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}: Loss {total_loss / len(dataloader):.4f}")

train(model, dataloader, epochs=70)

In [ ]:
# Denoising Function

def denoise(model, x_noisy, steps=T):
    x = x_noisy
    model.eval()
    for t in reversed(range(steps)):
        t_batch = torch.tensor([t] * x.size(0), device=device).long()
        with torch.no_grad():
            noise_pred = model(x, t_batch)
        beta = betas[t]
        alpha = alphas[t]
        alpha_bar = alphas_bar[t]
        x = (1 / torch.sqrt(alpha)) * (x - ((1 - alpha) / torch.sqrt(1 - alpha_bar)) * noise_pred)
        if t > 0:
            x += torch.sqrt(beta) * torch.randn_like(x)
    return x


In [ ]:
# Evaluation process

def evaluate_model(model, dataloader):
    model.eval()
    total_mse = 0
    total_ssim = 0
    count = 0

    for x, _ in dataloader:
        x = x.to(device)
        noisy, _ = forward_diffusion_sample(x, torch.full((x.size(0),), T - 1, device=device).long())
        denoised = denoise(model, noisy)

        x = x.clamp(0, 1)
        denoised = denoised.clamp(0, 1)

        for i in range(x.size(0)):
            gt = x[i][0].cpu().numpy()
            rec = denoised[i][0].detach().cpu().numpy()
            total_mse += ((gt - rec) ** 2).mean()
            total_ssim += ssim(gt, rec, data_range=1.0)
            count += 1
        if count >= 100:
            break

    print(f"Evaluation MSE: {total_mse / count:.4f}")
    print(f"Evaluation SSIM: {total_ssim / count:.4f}")

evaluate_model(model, dataloader)



In [ ]:
# Visualization of the dataset

def visualize(model, dataloader):
    model.eval()
    x, _ = next(iter(dataloader))
    x = x.to(device)[:6]
    noisy, _ = forward_diffusion_sample(x, torch.full((x.size(0),), T - 1, device=device).long())
    denoised = denoise(model, noisy)

    fig, axs = plt.subplots(3, 6, figsize=(12, 6))
    for i in range(6):
        axs[0, i].imshow(x[i][0].cpu(), cmap='gray')
        axs[0, i].set_title("Original")
        axs[0, i].axis('off')

        axs[1, i].imshow(noisy[i][0].cpu(), cmap='gray')
        axs[1, i].set_title("Noisy")
        axs[1, i].axis('off')

        axs[2, i].imshow(denoised[i][0].detach().cpu(), cmap='gray')
        axs[2, i].set_title("Denoised")
        axs[2, i].axis('off')

    plt.tight_layout()
    plt.show()

visualize(model, dataloader)
